In [41]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import rateslib as rl
import QuantLib as ql

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery, IRSwapStructure
from Query.IRSwaps.IRSwapStructure import IRSwapStructureFunctionMap
from Query.IRSwaps.IRSwapValue import IRSwapValue, IRSwapValueFunctionMap

# fmt: off
import Query.IRSwaps.adapter  # noqa: F401
# fmt: on

from utils.ql_utils import datetime_to_ql_date, ql_date_to_datetime 

# Fetch Curve

In [53]:
curve_mdp = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_sofr_mtv2_q12x11")
# curve_mdp = IRSwapsMDP(source="GSQUANT_RL")
# curve_mdp = IRSwapsMDP(source="CME_NY_EOD_LIVE-ql_basic")
# curve_mdp = IRSwapsMDP(source="CME_NY_EOD_LIVE-rl_basic")

In [54]:
curve = "USD-SOFR-1D"
timestamp = "live"
# timestamp = NY_tz.localize(datetime.datetime(2025, 10, 9, 17, 00))
# timestamp = datetime.date(2025, 10, 9)

curve_handle = curve_mdp._get_curve(curve_name=curve, timestamp=timestamp)

# risk model
imms = ["Z25", "H26", "M26", "U26", "Z26", "H27", "M27", "U27", "Z27", "H28", "M28", "U28", "Z28"]
sfrs = {}
for imm in imms:
    sfrs[imm] = rl.STIRFuture(
        effective=rl.scheduling.get_imm(code=imm), termination=rl.scheduling.next_imm(rl.scheduling.get_imm(code=imm)), spec="usd_stir", curves=curve_handle.handle()
    )

mt_tenors = ["5Y", "7Y", "10Y", "20Y", "30Y"]
# mt_tenors = ["5Y", "10Y", "30Y"]
mt_irs = {}
for t in mt_tenors:
    q = IRSwapQuery(curve="USD-SOFR-1D", tenor=t, structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1})
    pkg, _ = q.resolve_package(pricer_or_curve=curve_handle)
    mt_irs[t] = pkg[0]

rl_curve_risk_instruments = sfrs | mt_irs
rl_curve_risk_solver = rl.Solver(
    curves=[curve_handle.handle()],
    instruments=rl_curve_risk_instruments.values(),
    instrument_labels=rl_curve_risk_instruments.keys(),
    s=[r.rate().real for r in rl_curve_risk_instruments.values()],
    id=curve_handle.id(),
    func_tol=1e-8,
    conv_tol=1e-10,
)

FETCHING ERIS INTRADAY DISC CURVE...: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]


SUCCESS: `conv_tol` reached after 12 iterations (levenberg_marquardt), `f_val`: 0.0006652156434556791, `time`: 0.3628s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0036s


In [55]:
x_data_num, y_data_rate = curve_handle.handle()._plot_rates("1d", left=rl.NoInput(0), right=rl.NoInput(0))
plot_data_dict = dict(zip(x_data_num, [y.real for y in y_data_rate]))

calendar = ql.UnitedStates(ql.UnitedStates.FederalReserve)
filtered_plot_data = {dt: rate for dt, rate in plot_data_dict.items() if calendar.isBusinessDay(ql.Date(dt.day, dt.month, dt.year))}

x_business_days = list(filtered_plot_data.keys())
y_business_rates = list(filtered_plot_data.values())
curve_nodes = curve_handle.handle().nodes._nodes.keys()

# fig, ax = plt.subplots()
# ax.plot(x_business_days, y_business_rates)
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
# plt.xticks(rotation=45, ha="right")  # Rotate ticks for better readability
# ax.grid(True, linestyle="--", alpha=0.6)

# ticks = [t.tz_localize(None) if getattr(t, "tzinfo", None) else t for t in curve_nodes]
# ax.set_xticks(mdates.date2num(ticks))
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
# plt.title(f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]} | 1d")
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()


fig = go.Figure()
fig.add_trace(go.Scatter(x=x_business_days, y=y_business_rates, mode="lines", name="1D"))

tick_vals = [pd.Timestamp(t).tz_localize(None) for t in curve_nodes]
tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]

fig.update_layout(
    title=f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]} | 1d curve",
    template="plotly_dark",
    margin=dict(l=40, r=20, t=60, b=80),
    xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
    height=550,
    width=1200,
    yaxis=dict(showgrid=True),
)
fig.update_xaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikemode="across",
    showgrid=True,
)
fig.update_yaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikethickness=0.5,
    showgrid=True,
)

fig.show()

## Price Outright by tenor

In [17]:
risk = 25 
outright_query = IRSwapQuery(curve=curve, tenor="IMM_Z27xIMM_H28", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": risk})
outright_pkg, outright_rws = outright_query.resolve_package(pricer_or_curve=curve_handle) 

outright_vmap = outright_query.build_value_map(pricer_or_curve=curve_handle, package=outright_pkg, risk_weights=outright_rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.NOTIONAL, IRSwapValue.PV01]:
    print(v.name, outright_vmap.apply(value=v))

RATE 3.070502839169552
NPV -9.094947017729282e-13
NOTIONAL 1070466.6318352441
PV01 25.0


# Price Curve

In [8]:
risk = -100_000
curve_query = IRSwapQuery(curve=curve, structure=IRSwapStructure.CURVE, structure_kwargs={"front_tenor": "2Y", "back_tenor": "10Y", "bpv": risk})
curve_pkg, curve_rws = curve_query.resolve_package(pricer_or_curve=curve_handle)

curve_vmap = curve_query.build_value_map(pricer_or_curve=curve_handle, package=curve_pkg, risk_weights=curve_rws)
for v in [IRSwapValue.RATE, IRSwapValue.NPV, IRSwapValue.PV01, IRSwapValue.DV01]:
    print(v.name, curve_vmap.apply(value=v))

RATE -27.936950396657625
NPV 2.2351741790771484e-08
PV01 0.0
DV01 706.7074751220644


# Price Fly

In [38]:
# def _normalize_leg(s: str) -> str:
#     import re
#     # grab tenor tokens like 3M, 6m, 1Y, 2y (case-insensitive)
#     parts = re.findall(r'\d+\s*[dwmy]', s, flags=re.I)
#     parts = [p.upper().replace(" ", "") for p in parts]
#     if len(parts) == 1:
#         return parts[0]
#     if len(parts) == 2:
#         return f"{parts[0]}x{parts[1]}"
#     raise ValueError(f"Unexpected leg format: {s!r}")


# _normalize_leg("10y10y")

In [53]:
# flies = """ 
# 1y/1y1y/2y1y
# 1y1y/2y1y/3y1y
# 2y1/y3y1/4y1y
# 3y1y/4y1y/5y1y
# 4y1y/5y1y/6y1y
# 5y1y/6y1y/7y1y 
# """

In [61]:
# for fly_str in flies.split("\n"):
# 	if "/" not in fly_str:
# 		continue

# fly_str = "1y2Yy/1y5y/1y10y"
fly_str = "2y1y/3y1y/4y1y"
wing1, belly, wing2 = fly_str.split("/")

risk = -100_000
fly_query = IRSwapQuery(curve=curve, structure=IRSwapStructure.FLY, structure_kwargs={"front_tenor": wing1, "belly_tenor": belly, "back_tenor": wing2, "bpv": risk})
fly_pkg, fly_rws = fly_query.resolve_package(pricer_or_curve=curve_handle)

fly_vmap = fly_query.build_value_map(pricer_or_curve=curve_handle, package=fly_pkg, risk_weights=fly_rws)
print(fly_str, fly_vmap.apply(value=IRSwapValue.RATE))

display(rl.Portfolio(fly_pkg).delta(solver=rl_curve_risk_solver).style.format("{:_.0f}"))

2y1y/3y1y/4y1y -8.603058083615084


In [46]:
def rl_sfr(imm: str, risk=None, contracts=None):
    assert risk or contracts, "must pass in risk or contracts"
    if not contracts:
        contracts = -int(risk / 25)
    return rl.STIRFuture(
        effective=rl.scheduling.get_imm(code=imm),
        termination=rl.scheduling.next_imm(rl.scheduling.get_imm(code=imm)),
        spec="usd_stir",
        curves=curve_handle.handle(),
        contracts=contracts,
    )

In [56]:
rl_sfr(imm="Z27", contracts=100).rate()

<Dual: 3.097500, (live-SDR_INTRADAY-RL_USD_SOFR_MTV2_Q12x110, live-SDR_INTRADAY-RL_USD_SOFR_MTV2_Q12x111, live-SDR_INTRADAY-RL_USD_SOFR_MTV2_Q12x112, ...), [0.0, 0.0, 0.0, ...]>

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.stir_curve_building_utils import fetch_stir_market_data
_, sfrs, _ = fetch_stir_market_data("test", snap_local="live", fixings=None, side="mid", include_serff=False, n_ser_contracts=0, n_sfr_contracts=12, use_globex=False)

In [64]:
sfrs['SFRZ25'].fixed_rate.real

3.6424999999999983